In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/chess_data.zip"
!mkdir -p /content/dataset/
!unzip -q "{zip_path}" -d /content/dataset/

Mounted at /content/drive


In [ ]:
import pandas as pd
import glob

file_list = glob.glob("/content/dataset/chess_data/*.parquet")
print(f"Znaleziono {len(file_list)} plików Parquet.")

df = pd.concat([pd.read_parquet(f) for f in file_list], ignore_index=True)

print(f"Załadowano łącznie {len(df):,} pozycji szachowych.")
print(df['is_blunder'].value_counts(normalize=True))

Znaleziono 100 plików Parquet.
Załadowano łącznie 329,451 pozycji szachowych.
is_blunder
0    0.953125
1    0.046875
Name: proportion, dtype: float64


In [ ]:
!pip install chess
import numpy as np
import chess

def fen_to_tensor(fen):
    board = chess.Board(fen)
    # 8x8 plansza, 12 warstw (po 6 figur dla każdego koloru)
    tensor = np.zeros((8, 8, 12), dtype=np.float32)

    piece_map = {"P": 0, "N": 1, "B": 2, "R": 3, "Q": 4, "K": 5,
                 "p": 6, "n": 7, "b": 8, "r": 9, "q": 10, "k": 11}

    for square, piece in board.piece_map().items():
        row, col = divmod(square, 8)
        tensor[row, col, piece_map[piece.symbol()]] = 1
    return tensor

print("Konwersja FENów na macierze")
X = np.array([fen_to_tensor(f) for f in df['fen'].values], dtype=np.float32)
y = df['is_blunder'].values

print(f"Przygotowano X: {X.shape}, y: {y.shape}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 66.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=c1ccd4e66c4a006ee8870cba120cbd113962d95ba0ce0a883d39556aab09148f
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess
Konwersja FENów na macierze
Przygotowano X: (329451, 8, 8, 12), y: (329451,)


In [ ]:
from tensorflow.keras import layers, models
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42)

weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(weights))

model = models.Sequential([
    layers.Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(8, 8, 12)),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid') #
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 8, 8, 64)       │         6,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 605,889 (2.31 MB)

 Trainable params: 605,761 (2.31 MB)

 Non-trainable params: 128 (512.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=512,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=[early_stopping]
)

model.save('chess_complexity_model_v2.keras')
print("Model zapisany")

Epoch 1/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8164 - loss: 0.3721 - val_accuracy: 0.7993 - val_loss: 0.4303
Epoch 2/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8279 - loss: 0.3475 - val_accuracy: 0.7820 - val_loss: 0.4626
Epoch 3/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8321 - loss: 0.3337 - val_accuracy: 0.7879 - val_loss: 0.4469
Epoch 4/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8394 - loss: 0.3168 - val_accuracy: 0.7975 - val_loss: 0.4604
Epoch 5/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8460 - loss: 0.2984 - val_accuracy: 0.7731 - val_loss: 0.4801
Epoch 6/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8520 - loss: 0.2860 - val_accuracy: 0.7846 - val_loss: 0.4688
Epoch 7/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8563 - loss: 0.2777 - val_accuracy: 0.8124 - val_loss: 0.4457
Epoch 8/50
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8614 - loss: 0.2663 - val_accuracy: 0.